# Naive GLM vs Full Pipeline Comparison (Solidity)

Compares single-model naive generation (GLM 5.2, no RAG, no refinement, no kernel,
no spec-alignment, no security analysis) against the full MoE pipeline on the same
seed prompts (`sol_seed.jsonl` `description_long`).


In [ ]:
import json
import sys
import statistics
from math import sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Paths
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "nl2solidity"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

NAIVE_DIR = ROOT / "nl2solidity" / "dataset" / "naive_glm"
# Full-pipeline output lands in with_kernel_spec once batch_generate.py has run;
# dataset/data (curated reference corpus) is used as a fallback comparison set
# so this notebook is runnable before batch_generate.py has produced anything.
_WKS_DIR = ROOT / "nl2solidity" / "dataset" / "with_kernel_spec"
_DATA_DIR = ROOT / "nl2solidity" / "dataset" / "data"
PIPELINE_DIR = _WKS_DIR if any(_WKS_DIR.glob("*/meta.json")) else _DATA_DIR
print(f"Pipeline corpus: {PIPELINE_DIR.relative_to(ROOT)}")

# Find overlapping samples (live from filesystem)
pipe_ids = {p.parent.name for p in PIPELINE_DIR.glob("*/meta.json")}
naive_ids = {p.parent.name for p in NAIVE_DIR.glob("*/meta.json")} if NAIVE_DIR.exists() else set()
common = sorted(pipe_ids & naive_ids)
n = len(common)
assert n > 0, f"No overlapping samples found between {PIPELINE_DIR} and {NAIVE_DIR}"

from compiler_interface import check_code, is_compiler_available

def compile_breakdown(sol_path):
    code_txt = sol_path.read_text(encoding="utf-8").strip()
    if not code_txt or not is_compiler_available():
        return None
    r = check_code(code_txt)
    syn = sum(1 for e in r.errors if e.is_syntax_error())
    sem = sum(1 for e in r.errors if e.is_semantic_error())
    return {"is_valid": r.is_valid, "error_count": r.error_count,
            "syntax_error_count": syn, "semantic_error_count": sem}

# Recompiling is slow (one solc invocation per sample), so results are
# checkpointed after every sample. Re-running this cell reuses the checkpoint
# and only compiles what is missing; interrupt it freely.
COMPILE_CKPT = ROOT / "nl2solidity" / "dataset" / "compiler_per_sample.json"
FORCE_RECOMPILE = False   # True = ignore the checkpoint and recompile everything

def _load_compile_ckpt():
    if FORCE_RECOMPILE or not COMPILE_CKPT.exists():
        return {}
    try:
        return json.loads(COMPILE_CKPT.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        print(f"Checkpoint unreadable ({e}); starting fresh.")
        return {}

def _save_compile_ckpt(state):
    tmp = COMPILE_CKPT.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=1, sort_keys=True), encoding="utf-8")
    tmp.replace(COMPILE_CKPT)

cache = _load_compile_ckpt()
n_cached = sum(1 for sid in common if sid in cache)
print(f"Compiler checkpoint: {n_cached}/{len(common)} cached ({COMPILE_CKPT.name})")

per_sample = []
compile_stopped = False
for sid in common:
    if sid in cache:
        per_sample.append(cache[sid])
        continue
    nm = json.loads((NAIVE_DIR / sid / "meta.json").read_text(encoding="utf-8"))
    pm = json.loads((PIPELINE_DIR / sid / "meta.json").read_text(encoding="utf-8"))
    nv = nm.get("validation", {})
    pv = pm.get("validation", {})
    if "syntax_error_count" not in pv or "syntax_error_count" not in nv:
        try:
            if "syntax_error_count" not in pv:
                bd = compile_breakdown(PIPELINE_DIR / sid / f"{sid}.sol")
                if bd: pv = bd
            if "syntax_error_count" not in nv:
                bd = compile_breakdown(NAIVE_DIR / sid / f"{sid}.sol")
                if bd: nv = bd
        except KeyboardInterrupt:
            _save_compile_ckpt(cache)
            compile_stopped = True
            print(f"\n\u23f8  interrupted \u2014 {len(cache)} samples saved; "
                  "re-run this cell to resume.")
            break
    per_sample.append({
        "sid": sid,
        "naive": {
            "is_valid": nv.get("is_valid", False),
            "error_count": nv.get("error_count", 0),
            "syntax_error_count": nv.get("syntax_error_count", 0),
            "semantic_error_count": nv.get("semantic_error_count", 0),
        },
        "pipeline": {
            "is_valid": pv.get("is_valid", False),
            "error_count": pv.get("error_count", 0),
            "syntax_error_count": pv.get("syntax_error_count", 0),
            "semantic_error_count": pv.get("semantic_error_count", 0),
        },
    })
    cache[sid] = per_sample[-1]
    _save_compile_ckpt(cache)

if compile_stopped:
    print(f"Working with {len(per_sample)} of {len(common)} samples "
          "(plots below use whatever is loaded).")

n = len(per_sample)
assert n > 0, "no samples loaded"

def aggregate(samples, key):
    rows = [s[key] for s in samples]
    errs = [r["error_count"] for r in rows]
    valid_count = sum(1 for r in rows if r["is_valid"])
    syn_fail = sum(1 for r in rows if r["syntax_error_count"] > 0)
    sem_fail = sum(1 for r in rows if r["semantic_error_count"] > 0)
    nn = len(rows)
    return {
        "n": nn,
        "valid_rate": valid_count / nn * 100,
        "mean_errors": statistics.mean(errs) if errs else 0,
        "median_errors": statistics.median(errs) if errs else 0,
        "syntax_fail_rate": syn_fail / nn * 100,
        "semantic_fail_rate": sem_fail / nn * 100,
    }

naive = aggregate(per_sample, "naive")
pipeline = aggregate(per_sample, "pipeline")
delta = {
    "valid_rate_pp": pipeline["valid_rate"] - naive["valid_rate"],
    "mean_errors": pipeline["mean_errors"] - naive["mean_errors"],
    "syntax_fail_rate_pp": pipeline["syntax_fail_rate"] - naive["syntax_fail_rate"],
    "semantic_fail_rate_pp": pipeline["semantic_fail_rate"] - naive["semantic_fail_rate"],
}

labels = [f"Naive GLM\n(n={n})", f"Full Pipeline\n(n={n})"]
colors = ["#c44e52", "#4c72b0"]

try:
    from scipy.stats import t as student_t
    def _t_crit(nn):
        return float(student_t.ppf(0.975, nn - 1)) if nn > 1 else 0.0
except ImportError:
    def _t_crit(nn):
        return 1.96 if nn > 1 else 0.0

def ci95_mean(values):
    nn = len(values)
    if nn < 2: return 0.0
    se = np.std(values, ddof=1) / sqrt(nn)
    return _t_crit(nn) * se

def ci95_prop(flags):
    nn = len(flags)
    if nn < 2: return 0.0
    p = float(np.mean(flags))
    se = sqrt(p * (1 - p) / nn)
    return _t_crit(nn) * se * 100

print(f"Loaded {n} paired samples "
      f"({n_cached} from checkpoint, {n - n_cached} freshly compiled)")


In [ ]:
# Valid rate and mean errors (bar charts with 95% CI)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = np.arange(2)

valid_rates = [naive["valid_rate"], pipeline["valid_rate"]]
naive_valid_flags = [s["naive"]["is_valid"] for s in per_sample]
pipe_valid_flags = [s["pipeline"]["is_valid"] for s in per_sample]
valid_err = [ci95_prop(naive_valid_flags), ci95_prop(pipe_valid_flags)]

axes[0].bar(x, valid_rates, yerr=valid_err, capsize=4, color=colors, ecolor="black", linewidth=0.8)
axes[0].set_xticks(x, labels)
axes[0].set_ylabel("Valid rate (%)")
axes[0].set_title("solc-valid outputs (95% CI)")
ymax = max(valid_rates[i] + valid_err[i] for i in range(2)) if any(valid_rates) else 10
axes[0].set_ylim(0, max(10, ymax * 1.15 + 1))

mean_errors = [naive["mean_errors"], pipeline["mean_errors"]]
naive_errs = [s["naive"]["error_count"] for s in per_sample]
pipe_errs = [s["pipeline"]["error_count"] for s in per_sample]
mean_err = [ci95_mean(naive_errs), ci95_mean(pipe_errs)]

axes[1].bar(x, mean_errors, yerr=mean_err, capsize=4, color=colors, ecolor="black", linewidth=0.8)
axes[1].set_xticks(x, labels)
axes[1].set_ylabel("Mean solc errors")
axes[1].set_title("Mean errors per sample (95% CI)")

plt.tight_layout()
plt.show()


In [ ]:
# Error distribution (box plot)
fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot(
    [naive_errs, pipe_errs],
    tick_labels=labels,
    patch_artist=True,
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel("solc errors")
ax.set_title(f"Error distribution per sample (n={n} each)")
plt.tight_layout()
plt.show()


In [ ]:
# Pairwise delta (horizontal bar)
metrics = ["valid_rate_pp", "mean_errors", "syntax_fail_rate_pp", "semantic_fail_rate_pp"]
metric_labels = ["\u0394 valid rate (pp)", "\u0394 mean errors", "\u0394 syntax fail (pp)", "\u0394 semantic fail (pp)"]
values = [delta[m] for m in metrics]

fig, ax = plt.subplots(figsize=(8, 3.5))
bar_colors = ["#55a868" if (v > 0 and "valid" in m) or (v < 0 and "error" in m) or (v < 0 and "fail" in m)
              else "#c44e52" for v, m in zip(values, metrics)]
ax.barh(metric_labels, values, color=bar_colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pipeline minus Naive")
ax.set_title("Full Pipeline vs Naive GLM \u2014 Improvement")
plt.tight_layout()
plt.show()


In [ ]:
# Per-sample scatter: naive errors vs pipeline errors
fig, ax = plt.subplots(figsize=(6, 6))
xs = [s["naive"]["error_count"] for s in per_sample]
ys = [s["pipeline"]["error_count"] for s in per_sample]

ax.scatter(xs, ys, alpha=0.7, edgecolors="black", linewidths=0.5, s=60)
max_val = max(max(xs), max(ys)) * 1.1 if (max(xs) or max(ys)) else 1
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.4, label="y = x (no improvement)")
ax.set_xlabel("Naive GLM errors")
ax.set_ylabel("Full Pipeline errors")
ax.set_title("Per-sample error comparison")
ax.legend()
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
plt.tight_layout()
plt.show()


In [ ]:
# Summary table
print(f"{'Metric':<25} {'Naive GLM':>12} {'Full Pipeline':>14} {'Delta':>10}")
print("-" * 65)
print(f"{'Valid rate (%)':<25} {naive['valid_rate']:>12.1f} {pipeline['valid_rate']:>14.1f} {delta['valid_rate_pp']:>+10.1f}")
print(f"{'Mean errors':<25} {naive['mean_errors']:>12.1f} {pipeline['mean_errors']:>14.1f} {delta['mean_errors']:>+10.1f}")
print(f"{'Median errors':<25} {naive['median_errors']:>12.1f} {pipeline['median_errors']:>14.1f} {pipeline['median_errors'] - naive['median_errors']:>+10.1f}")
print(f"{'Syntax fail rate (%)':<25} {naive['syntax_fail_rate']:>12.1f} {pipeline['syntax_fail_rate']:>14.1f} {delta['syntax_fail_rate_pp']:>+10.1f}")
print(f"{'Semantic fail rate (%)':<25} {naive['semantic_fail_rate']:>12.1f} {pipeline['semantic_fail_rate']:>14.1f} {delta['semantic_fail_rate_pp']:>+10.1f}")


---
# Security Analysis Evaluation (Slither)

Runs static security analysis (`security_analysis.analyze_solidity`, backed by
Slither) on each sample and compares finding counts, actionable (High/Medium,
non-low-confidence) findings, and the impact-severity breakdown for Naive GLM
vs Full Pipeline.

**This cell is self-contained** \u2014 it loads data independently and can be run
without the cells above.


In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "nl2solidity"))

NAIVE_DIR = ROOT / "nl2solidity" / "dataset" / "naive_glm"
_WKS_DIR = ROOT / "nl2solidity" / "dataset" / "with_kernel_spec"
_DATA_DIR = ROOT / "nl2solidity" / "dataset" / "data"
PIPELINE_DIR = _WKS_DIR if any(_WKS_DIR.glob("*/meta.json")) else _DATA_DIR

pipe_ids = {p.parent.name for p in PIPELINE_DIR.glob("*/meta.json")}
naive_ids = {p.parent.name for p in NAIVE_DIR.glob("*/meta.json")} if NAIVE_DIR.exists() else set()
common = sorted(pipe_ids & naive_ids)
print(f"Overlapping samples: {len(common)}")

from security_analysis import analyze_solidity, summarize, is_analysis_available

# Resume checkpoints: one file per corpus, rewritten after every sample.
# Slither is slow, so interrupt this cell freely.
SEC_CKPT = {"naive": ROOT / "nl2solidity" / "dataset" / "security_results_naive.json",
            "pipeline": ROOT / "nl2solidity" / "dataset" / "security_results_pipeline.json"}
RETRY_ERRORS = True
FORCE_RERUN = False

def _load_ckpt(path):
    if FORCE_RERUN or not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        print(f"  checkpoint {path.name} unreadable ({e}); starting fresh.")
        return {}

def _save_ckpt(path, state):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=1, sort_keys=True), encoding="utf-8")
    tmp.replace(path)

def _done(rec):
    return rec is not None and not (RETRY_ERRORS and "error" in rec)

def run_security_on(directory, sample_ids, ckpt_path):
    results = _load_ckpt(ckpt_path)
    todo = [s for s in sample_ids if not _done(results.get(s))]
    print(f"  checkpoint: {len(results)} recorded, {len(todo)} to run ({ckpt_path.name})")
    for i, sid in enumerate(todo, 1):
        sol_path = directory / sid / f"{sid}.sol"
        code_txt = sol_path.read_text(encoding="utf-8").strip() if sol_path.exists() else ""
        if not code_txt:
            results[sid] = {"n_findings": 0, "n_actionable": 0, "by_impact": {}, "empty": True}
            _save_ckpt(ckpt_path, results)
            print(f"  [{i}/{len(todo)}] {sid}: EMPTY")
            continue
        try:
            summary = summarize(analyze_solidity(code_txt))
            results[sid] = {
                "available": summary.get("available", False),
                "analyzed": summary.get("analyzed", False),
                "n_findings": summary.get("n_findings", 0),
                "n_actionable": summary.get("n_actionable", 0),
                "by_impact": summary.get("by_impact", {}),
            }
        except KeyboardInterrupt:
            _save_ckpt(ckpt_path, results)
            print(f"\n  \u23f8  interrupted after {i - 1} of {len(todo)} \u2014 progress saved.")
            return results, True
        except Exception as e:
            results[sid] = {"n_findings": 0, "n_actionable": 0, "by_impact": {},
                            "error": f"{type(e).__name__}: {e}"[:300]}
        _save_ckpt(ckpt_path, results)
        r = results[sid]
        print(f"  [{i}/{len(todo)}] {sid}: findings={r['n_findings']} actionable={r['n_actionable']}")
    _save_ckpt(ckpt_path, results)
    return results, False

if not is_analysis_available():
    print("\u26a0 No static analyzer available (pip install slither-analyzer) \u2014 "
          "this section will show all-zero findings.")

print("\n--- Running security analysis on NAIVE samples ---")
naive_sec_results, naive_stopped = run_security_on(NAIVE_DIR, common, SEC_CKPT["naive"])

if naive_stopped:
    pipe_sec_results, pipe_stopped = _load_ckpt(SEC_CKPT["pipeline"]), True
    print("\n--- PIPELINE samples: not started (interrupted above) ---")
else:
    print("\n--- Running security analysis on PIPELINE samples ---")
    pipe_sec_results, pipe_stopped = run_security_on(PIPELINE_DIR, common, SEC_CKPT["pipeline"])

security_common = [s for s in common if s in naive_sec_results and s in pipe_sec_results]
if naive_stopped or pipe_stopped:
    print(f"\n\u23f8  STOPPED \u2014 re-run this cell to resume "
          f"({len(security_common)}/{len(common)} paired so far).")

n = len(security_common)
naive_findings = [naive_sec_results[s]["n_findings"] for s in security_common]
pipe_findings = [pipe_sec_results[s]["n_findings"] for s in security_common]
naive_actionable = [naive_sec_results[s]["n_actionable"] for s in security_common]
pipe_actionable = [pipe_sec_results[s]["n_actionable"] for s in security_common]

print(f"\n{'='*60}")
print(f"Paired samples with security results: {n}/{len(common)}")
if n:
    print(f"Mean findings   \u2014 Naive: {np.mean(naive_findings):.2f}  Pipeline: {np.mean(pipe_findings):.2f}")
    print(f"Mean actionable \u2014 Naive: {np.mean(naive_actionable):.2f}  Pipeline: {np.mean(pipe_actionable):.2f}")
    clean_naive = sum(1 for x in naive_actionable if x == 0)
    clean_pipe = sum(1 for x in pipe_actionable if x == 0)
    print(f"Actionable-clean rate \u2014 Naive: {clean_naive/n*100:.1f}%  Pipeline: {clean_pipe/n*100:.1f}%")


In [ ]:
# --- Security visualizations ---
lbl = [f"Naive GLM\n(n={n})", f"Full Pipeline\n(n={n})"]
colors = ["#c44e52", "#4c72b0"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x = np.arange(2)

means_f = [np.mean(naive_findings) if naive_findings else 0, np.mean(pipe_findings) if pipe_findings else 0]
axes[0].bar(x, means_f, color=colors, edgecolor="black")
axes[0].set_xticks(x, lbl)
axes[0].set_ylabel("Mean findings per sample")
axes[0].set_title("Slither findings (all severities)")

means_a = [np.mean(naive_actionable) if naive_actionable else 0, np.mean(pipe_actionable) if pipe_actionable else 0]
axes[1].bar(x, means_a, color=colors, edgecolor="black")
axes[1].set_xticks(x, lbl)
axes[1].set_ylabel("Mean actionable findings per sample")
axes[1].set_title("Actionable (High/Medium) findings")
plt.tight_layout()
plt.show()

# Impact-severity breakdown (stacked-style grouped bars)
impacts = ["High", "Medium", "Low", "Informational", "Optimization"]
naive_by_impact = Counter()
pipe_by_impact = Counter()
for s in security_common:
    naive_by_impact.update(naive_sec_results[s].get("by_impact", {}))
    pipe_by_impact.update(pipe_sec_results[s].get("by_impact", {}))

present = [i for i in impacts if naive_by_impact.get(i, 0) or pipe_by_impact.get(i, 0)]
present += sorted(set(naive_by_impact) | set(pipe_by_impact)) if not present else []
present = list(dict.fromkeys(present))
if present:
    y = np.arange(len(present))
    h = 0.35
    fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(present))))
    ax.barh(y + h/2, [naive_by_impact.get(i, 0) for i in present], height=h, color=colors[0], label="Naive GLM")
    ax.barh(y - h/2, [pipe_by_impact.get(i, 0) for i in present], height=h, color=colors[1], label="Full Pipeline")
    ax.set_yticks(y, present)
    ax.set_xlabel("Total findings across corpus")
    ax.set_title("Findings by Impact Severity")
    ax.legend()
    plt.tight_layout()
    plt.show()

# Per-sample scatter: naive actionable vs pipeline actionable
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(naive_actionable, pipe_actionable, alpha=0.7, edgecolors="black", linewidths=0.5, s=60)
max_val = max(max(naive_actionable, default=0), max(pipe_actionable, default=0)) * 1.1 + 1
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.4, label="y = x (no improvement)")
ax.set_xlabel("Naive GLM actionable findings")
ax.set_ylabel("Full Pipeline actionable findings")
ax.set_title("Per-sample Actionable Findings Comparison")
ax.legend()
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
plt.tight_layout()
plt.show()


---
# Execution Evaluation (Foundry / forge)

Builds a minimal Foundry harness for each sample (`solidity_execution.run_solidity_execution`,
fuzz tier only \u2014 no LLM-authored property tests, since naive samples do not have any)
and compares deploy/build success and per-sample defect counts for Naive GLM vs Full
Pipeline.

**This cell is self-contained** \u2014 it loads data independently and can be run
without the cells above. Foundry execution is slow; interrupt freely and re-run
to resume from the checkpoint.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "nl2solidity"))

NAIVE_DIR = ROOT / "nl2solidity" / "dataset" / "naive_glm"
_WKS_DIR = ROOT / "nl2solidity" / "dataset" / "with_kernel_spec"
_DATA_DIR = ROOT / "nl2solidity" / "dataset" / "data"
PIPELINE_DIR = _WKS_DIR if any(_WKS_DIR.glob("*/meta.json")) else _DATA_DIR

pipe_ids = {p.parent.name for p in PIPELINE_DIR.glob("*/meta.json")}
naive_ids = {p.parent.name for p in NAIVE_DIR.glob("*/meta.json")} if NAIVE_DIR.exists() else set()
common = sorted(pipe_ids & naive_ids)
print(f"Overlapping samples: {len(common)}")

from solidity_execution import ExecutionRequest, run_solidity_execution, is_runner_available

EXEC_CKPT = {"naive": ROOT / "nl2solidity" / "dataset" / "exec_results_naive.json",
             "pipeline": ROOT / "nl2solidity" / "dataset" / "exec_results_pipeline.json"}
RETRY_ERRORS = True
FORCE_RERUN = False

def _load_ckpt(path):
    if FORCE_RERUN or not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        print(f"  checkpoint {path.name} unreadable ({e}); starting fresh.")
        return {}

def _save_ckpt(path, state):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=1, sort_keys=True), encoding="utf-8")
    tmp.replace(path)

def _done(rec):
    return rec is not None and not (RETRY_ERRORS and "error" in rec)

def run_exec_on(directory, sample_ids, ckpt_path):
    results = _load_ckpt(ckpt_path)
    todo = [s for s in sample_ids if not _done(results.get(s))]
    print(f"  checkpoint: {len(results)} recorded, {len(todo)} to run ({ckpt_path.name})")
    for i, sid in enumerate(todo, 1):
        sol_path = directory / sid / f"{sid}.sol"
        code_txt = sol_path.read_text(encoding="utf-8").strip() if sol_path.exists() else ""
        if not code_txt:
            results[sid] = {"success": False, "compiled": False, "n_failures": 0, "empty": True}
            _save_ckpt(ckpt_path, results)
            print(f"  [{i}/{len(todo)}] {sid}: EMPTY")
            continue
        try:
            req = ExecutionRequest(candidate_solidity=code_txt, tiers=["fuzz"])
            result = run_solidity_execution(req)
            results[sid] = {
                "success": bool(result.success),
                "compiled": bool(result.compiled),
                "n_failures": len(result.failures()),
                "tier_status": dict(result.tier_status or {}),
            }
        except KeyboardInterrupt:
            _save_ckpt(ckpt_path, results)
            print(f"\n  \u23f8  interrupted after {i - 1} of {len(todo)} \u2014 progress saved.")
            return results, True
        except Exception as e:
            results[sid] = {"success": False, "compiled": False, "n_failures": 0,
                            "error": f"{type(e).__name__}: {e}"[:300]}
        _save_ckpt(ckpt_path, results)
        r = results[sid]
        status = "PASS" if r["success"] else ("COMPILED" if r.get("compiled") else "FAIL")
        print(f"  [{i}/{len(todo)}] {sid}: {status} ({r['n_failures']} defects)")
    _save_ckpt(ckpt_path, results)
    return results, False

if not is_runner_available():
    print("\u26a0 forge/forge-std not available \u2014 execution results will be all-fail.")

print("\n--- Running execution on NAIVE samples ---")
naive_exec_results, naive_stopped = run_exec_on(NAIVE_DIR, common, EXEC_CKPT["naive"])

if naive_stopped:
    pipe_exec_results, pipe_stopped = _load_ckpt(EXEC_CKPT["pipeline"]), True
    print("\n--- PIPELINE samples: not started (interrupted above) ---")
else:
    print("\n--- Running execution on PIPELINE samples ---")
    pipe_exec_results, pipe_stopped = run_exec_on(PIPELINE_DIR, common, EXEC_CKPT["pipeline"])

exec_common = [s for s in common if s in naive_exec_results and s in pipe_exec_results]
if naive_stopped or pipe_stopped:
    print(f"\n\u23f8  STOPPED \u2014 re-run this cell to resume "
          f"({len(exec_common)}/{len(common)} paired so far).")

n = len(exec_common)
naive_pass = sum(1 for s in exec_common if naive_exec_results[s].get("success"))
pipe_pass = sum(1 for s in exec_common if pipe_exec_results[s].get("success"))
naive_compiled = sum(1 for s in exec_common if naive_exec_results[s].get("compiled"))
pipe_compiled = sum(1 for s in exec_common if pipe_exec_results[s].get("compiled"))

print(f"\n{'='*60}")
print(f"Paired samples with execution results: {n}/{len(common)}")
if n:
    print(f"Deploy-compiled rate \u2014 Naive: {naive_compiled/n*100:.1f}%  Pipeline: {pipe_compiled/n*100:.1f}%")
    print(f"Fuzz-pass rate       \u2014 Naive: {naive_pass/n*100:.1f}%  Pipeline: {pipe_pass/n*100:.1f}%")


In [ ]:
# --- Execution visualizations ---
lbl = [f"Naive GLM\n(n={n})", f"Full Pipeline\n(n={n})"]
colors = ["#c44e52", "#4c72b0"]

naive_failures = [naive_exec_results[s].get("n_failures", 0) for s in exec_common]
pipe_failures = [pipe_exec_results[s].get("n_failures", 0) for s in exec_common]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.arange(2)

axes[0].bar(x, [naive_compiled/n*100 if n else 0, pipe_compiled/n*100 if n else 0], color=colors, edgecolor="black")
axes[0].set_xticks(x, lbl)
axes[0].set_ylabel("Rate (%)")
axes[0].set_title("Foundry deploy-compiled rate")
axes[0].set_ylim(0, 105)

axes[1].bar(x, [naive_pass/n*100 if n else 0, pipe_pass/n*100 if n else 0], color=colors, edgecolor="black")
axes[1].set_xticks(x, lbl)
axes[1].set_ylabel("Rate (%)")
axes[1].set_title("Fuzz-tier pass rate")
axes[1].set_ylim(0, 105)

axes[2].bar(x, [np.mean(naive_failures) if naive_failures else 0, np.mean(pipe_failures) if pipe_failures else 0],
            color=colors, edgecolor="black")
axes[2].set_xticks(x, lbl)
axes[2].set_ylabel("Mean defects per sample")
axes[2].set_title("Contract-level execution defects")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(naive_failures, pipe_failures, alpha=0.7, edgecolors="black", linewidths=0.5, s=60)
max_val = max(max(naive_failures, default=0), max(pipe_failures, default=0)) * 1.1 + 1
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.4, label="y = x (no improvement)")
ax.set_xlabel("Naive GLM execution defects")
ax.set_ylabel("Full Pipeline execution defects")
ax.set_title("Per-sample Execution Defect Comparison")
ax.legend()
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
plt.tight_layout()
plt.show()


---
# Spec-Alignment (Semantic Similarity) Evaluation

Runs the spec-aligner (`compare_pair`, Solidity question bank
`spec_aligner/questions_solidity.json`, same `profile="runtime", shards=3`
settings the full pipeline's quality gate uses) on each naive sample against
its NL prompt, then compares similarity scores with the full pipeline's
stored `spec_alignment.similarity`.

**This cell is self-contained** \u2014 it loads data independently and can be run
without the cells above.

\u26a0\ufe0f This cell makes LLM API calls (one scoring per sample).


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "nl2solidity"))
sys.path.insert(0, str(ROOT))  # spec_aligner lives at the repo root

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

NAIVE_DIR = ROOT / "nl2solidity" / "dataset" / "naive_glm"
_WKS_DIR = ROOT / "nl2solidity" / "dataset" / "with_kernel_spec"
_DATA_DIR = ROOT / "nl2solidity" / "dataset" / "data"
PIPELINE_DIR = _WKS_DIR if any(_WKS_DIR.glob("*/meta.json")) else _DATA_DIR

CKPT = ROOT / "nl2solidity" / "dataset" / "spec_align_naive_scores.json"
RETRY_ERRORS = True
FORCE_RESCORE = False

pipe_ids = {p.parent.name for p in PIPELINE_DIR.glob("*/meta.json")}
naive_ids = {p.parent.name for p in NAIVE_DIR.glob("*/meta.json")} if NAIVE_DIR.exists() else set()
common = sorted(pipe_ids & naive_ids)
print(f"Overlapping samples: {len(common)}")

from spec_aligner.pipeline import compare_pair
from spec_aligner.bank import SOLIDITY_BANK_PATH
from spec_aligner.llm import ask, CliUsageLimitError

def load_ckpt():
    if FORCE_RESCORE or not CKPT.exists():
        return {}
    try:
        return json.loads(CKPT.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as e:
        print(f"Checkpoint unreadable ({e}); starting fresh.")
        return {}

def save_ckpt(state):
    tmp = CKPT.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=1, sort_keys=True), encoding="utf-8")
    tmp.replace(CKPT)

state = load_ckpt()

def is_done(sid):
    rec = state.get(sid)
    if rec is None:
        return False
    return not (RETRY_ERRORS and rec.get("status") == "error")

todo = [sid for sid in common if not is_done(sid)]
print(f"Checkpoint: {len(state)} recorded, {len(todo)} to score "
      f"({CKPT.relative_to(ROOT)})")

stopped = None
for i, sid in enumerate(todo, 1):
    nl_path = NAIVE_DIR / sid / f"{sid}.txt"
    sol_path = NAIVE_DIR / sid / f"{sid}.sol"
    nl = nl_path.read_text(encoding="utf-8").strip() if nl_path.exists() else ""
    sol = sol_path.read_text(encoding="utf-8").strip() if sol_path.exists() else ""
    if not nl or not sol:
        state[sid] = {"similarity": None, "status": "empty"}
        save_ckpt(state)
        print(f"  [{i}/{len(todo)}] {sid}: SKIPPED (empty)")
        continue
    try:
        result = compare_pair(nl, sol, ask, sample_id=f"naive-{sid}",
                              profile="runtime", shards=3, bank_path=SOLIDITY_BANK_PATH)
        sim = result.get("summary", {}).get("similarity")
        state[sid] = {"similarity": sim, "status": "ok"}
        print(f"  [{i}/{len(todo)}] {sid}: similarity={sim}")
    except KeyboardInterrupt:
        stopped = "interrupted by user"
        break
    except CliUsageLimitError as e:
        state[sid] = {"similarity": None, "status": "error", "error": str(e)[:300]}
        save_ckpt(state)
        stopped = f"provider usage limit: {e}"
        break
    except Exception as e:
        state[sid] = {"similarity": None, "status": "error", "error": f"{type(e).__name__}: {e}"[:300]}
        print(f"  [{i}/{len(todo)}] {sid}: ERROR \u2014 {e}")
    save_ckpt(state)

save_ckpt(state)
if stopped:
    remaining = [s for s in common if not is_done(s)]
    print(f"\n\u23f8  STOPPED \u2014 {stopped}")
    print(f"   Progress saved: {len(state)} of {len(common)} recorded, "
          f"{len(remaining)} left. Re-run this cell to resume.")
    print("   The plotting cell below works on whatever has been scored so far.")

naive_similarities = {sid: state.get(sid, {}).get("similarity") for sid in common}
n_err = sum(1 for sid in common if state.get(sid, {}).get("status") == "error")
if n_err:
    print(f"\n{n_err} sample(s) errored; re-run the cell to retry them "
          f"(RETRY_ERRORS={RETRY_ERRORS}).")

n_ok = sum(1 for sid in common if state.get(sid, {}).get("status") == "ok")
print(f"\nCheckpoint holds {n_ok} scored sample(s) \u2014 run the next cell to plot them.")


In [ ]:
# --- Spec-alignment plots (reads the checkpoint; safe to run mid-scoring) ---
import json
import sys
from math import sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
NAIVE_DIR = ROOT / "nl2solidity" / "dataset" / "naive_glm"
_WKS_DIR = ROOT / "nl2solidity" / "dataset" / "with_kernel_spec"
_DATA_DIR = ROOT / "nl2solidity" / "dataset" / "data"
PIPELINE_DIR = _WKS_DIR if any(_WKS_DIR.glob("*/meta.json")) else _DATA_DIR
CKPT = ROOT / "nl2solidity" / "dataset" / "spec_align_naive_scores.json"

pipe_ids = {p.parent.name for p in PIPELINE_DIR.glob("*/meta.json")}
naive_ids = {p.parent.name for p in NAIVE_DIR.glob("*/meta.json")} if NAIVE_DIR.exists() else set()
common = sorted(pipe_ids & naive_ids)

if not CKPT.exists():
    raise SystemExit(f"No checkpoint at {CKPT} \u2014 run the scoring cell above first.")
state = json.loads(CKPT.read_text(encoding="utf-8"))

naive_similarities = {sid: state.get(sid, {}).get("similarity") for sid in common}
n_err = sum(1 for sid in common if state.get(sid, {}).get("status") == "error")
n_todo = sum(1 for sid in common if sid not in state)
print(f"Checkpoint: {len(state)}/{len(common)} recorded"
      + (f", {n_todo} not yet scored" if n_todo else "")
      + (f", {n_err} errored" if n_err else ""))

pipe_similarities = {}
for sid in common:
    meta = json.loads((PIPELINE_DIR / sid / "meta.json").read_text(encoding="utf-8"))
    sim = meta.get("spec_alignment", {}).get("similarity")
    pipe_similarities[sid] = sim

scored = [(sid, naive_similarities[sid], pipe_similarities[sid])
          for sid in common
          if naive_similarities.get(sid) is not None and pipe_similarities.get(sid) is not None]
print(f"Plotting {len(scored)}/{len(common)} samples with both scores")

if not scored:
    raise SystemExit("No samples scored yet on both sides \u2014 run the scoring cell above, "
                      "and confirm the pipeline corpus has spec_alignment in meta.json.")

naive_sims = [s[1] for s in scored]
pipe_sims = [s[2] for s in scored]

print(f"\nMean similarity \u2014 Naive: {np.mean(naive_sims):.4f}")
print(f"Mean similarity \u2014 Pipeline: {np.mean(pipe_sims):.4f}")
print(f"Delta: {np.mean(pipe_sims) - np.mean(naive_sims):+.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(naive_sims, pipe_sims, alpha=0.7, edgecolors="black", linewidths=0.5, s=60)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="y = x")
ax.set_xlabel("Naive GLM similarity")
ax.set_ylabel("Full Pipeline similarity")
partial = " (partial)" if n_todo or n_err else ""
ax.set_title(f"Spec-Alignment Similarity Comparison{partial}")
ax.legend()
ax.set_xlim(0.4, 1.0)
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
means = [np.mean(naive_sims), np.mean(pipe_sims)]
try:
    from scipy.stats import t as student_t
    def _t_crit(n):
        return float(student_t.ppf(0.975, n - 1)) if n > 1 else 0.0
except ImportError:
    def _t_crit(n):
        return 1.96 if n > 1 else 0.0
def _ci95(vals):
    n = len(vals)
    if n < 2:
        return 0.0
    return _t_crit(n) * np.std(vals, ddof=1) / sqrt(n)

errs = [_ci95(naive_sims), _ci95(pipe_sims)]

n_scored = len(scored)
ax.bar([f"Naive GLM\n(n={n_scored})", f"Full Pipeline\n(n={n_scored})"], means,
       yerr=errs, capsize=4, color=["#c44e52", "#4c72b0"], ecolor="black")
ax.set_ylabel("Mean similarity")
ax.set_title(f"Spec-Alignment Semantic Similarity (95% CI){partial}")
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()


---
# Power Analysis & Statistical Soundness

Effect sizes, exact power ($1-\beta$), and paired significance tests for every metric above.
Self-contained for the compiler, security, execution, and spec-alignment metrics.

**Why these tests.** The two corpora are **paired** \u2014 the same $n$ prompts generated by
both the naive single-model path and the full pipeline \u2014 so each sample is its own
control, and tests that exploit pairing (McNemar for pass/fail, Wilcoxon signed-rank
for counts/similarity) are far more sensitive than unpaired alternatives at the same $n$.


In [ ]:
# --- Power analysis & paired significance tests -------------------------------
import json
import sys
from math import asin, comb, sqrt
from pathlib import Path
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "nl2solidity"))
sys.path.insert(0, str(ROOT))

NAIVE_DIR = ROOT / "nl2solidity" / "dataset" / "naive_glm"
_WKS_DIR = ROOT / "nl2solidity" / "dataset" / "with_kernel_spec"
_DATA_DIR = ROOT / "nl2solidity" / "dataset" / "data"
PIPELINE_DIR = _WKS_DIR if any(_WKS_DIR.glob("*/meta.json")) else _DATA_DIR
CKPT = ROOT / "nl2solidity" / "dataset" / "spec_align_naive_scores.json"

ALPHA = 0.05
_N = NormalDist()
Z_CRIT = _N.inv_cdf(1 - ALPHA / 2)

pipe_ids = {p.parent.name for p in PIPELINE_DIR.glob("*/meta.json")}
naive_ids = {p.parent.name for p in NAIVE_DIR.glob("*/meta.json")} if NAIVE_DIR.exists() else set()
common = sorted(pipe_ids & naive_ids)
n = len(common)
print(f"Paired samples: n = {n}   alpha = {ALPHA}")


def _p_from_z(z):
    p = 2 * _N.cdf(-abs(z))
    return max(p, 1e-300)


def fmt_p(p):
    return "< 1e-300" if p <= 1e-300 else (f"{p:.3g}" if p >= 1e-4 else f"{p:.2e}")


def cohens_h(p1, p2):
    return 2 * asin(sqrt(p1)) - 2 * asin(sqrt(p2))


def mcnemar(a_flags, b_flags):
    b = sum(1 for x, y in zip(a_flags, b_flags) if x and not y)
    c = sum(1 for x, y in zip(a_flags, b_flags) if y and not x)
    nd = b + c
    if nd == 0:
        return {"b": b, "c": c, "n_discordant": 0, "p": 1.0, "z": 0.0, "exact": True}
    if nd <= 1000:
        k = min(b, c)
        tail = sum(comb(nd, i) for i in range(k + 1)) / (2 ** nd)
        p, exact = min(1.0, 2 * tail), True
    else:
        p, exact = None, False
    z = (abs(c - b) - 1) / sqrt(nd)
    if p is None:
        p = _p_from_z(z)
    return {"b": b, "c": c, "n_discordant": nd, "p": p, "z": z, "exact": exact}


def _ranks(vals):
    order = np.argsort(vals, kind="mergesort")
    r = np.empty(len(vals), dtype=float)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and vals[order[j + 1]] == vals[order[i]]:
            j += 1
        r[order[i:j + 1]] = (i + j) / 2 + 1
        i = j + 1
    return r


def wilcoxon(diffs):
    d = np.asarray([x for x in diffs if x != 0], dtype=float)
    nr = len(d)
    if nr == 0:
        return {"n_nonzero": 0, "p": 1.0, "z": 0.0, "rb": 0.0, "w_plus": 0.0, "w_minus": 0.0}
    r = _ranks(np.abs(d))
    w_plus = float(r[d > 0].sum())
    w_minus = float(r[d < 0].sum())
    mu = nr * (nr + 1) / 4
    _, counts = np.unique(np.abs(d), return_counts=True)
    tie_corr = float(sum(t ** 3 - t for t in counts))
    var = nr * (nr + 1) * (2 * nr + 1) / 24 - tie_corr / 48
    z = (w_plus - mu - np.sign(w_plus - mu) * 0.5) / sqrt(var) if var > 0 else 0.0
    rb = (w_plus - w_minus) / (w_plus + w_minus) if (w_plus + w_minus) else 0.0
    return {"n_nonzero": nr, "p": _p_from_z(z), "z": float(z), "rb": float(rb),
            "w_plus": w_plus, "w_minus": w_minus}


def cohens_dz(diffs):
    d = np.asarray(diffs, dtype=float)
    sd = d.std(ddof=1)
    return float(d.mean() / sd) if sd > 0 else 0.0


def power_two_proportions(h, n_per_group, alpha=ALPHA):
    lam = abs(h) * sqrt(n_per_group / 2)
    zc = _N.inv_cdf(1 - alpha / 2)
    return _N.cdf(lam - zc) + _N.cdf(-lam - zc)


def power_mcnemar(b, c, alpha=ALPHA):
    nd = b + c
    if nd == 0:
        return 0.0
    p_disc = c / nd
    lam = sqrt(nd) * abs(2 * p_disc - 1)
    zc = _N.inv_cdf(1 - alpha / 2)
    return _N.cdf(lam - zc) + _N.cdf(-lam - zc)


def power_paired(dz, n_pairs, alpha=ALPHA):
    lam = abs(dz) * sqrt(n_pairs)
    zc = _N.inv_cdf(1 - alpha / 2)
    return _N.cdf(lam - zc) + _N.cdf(-lam - zc)


def n_for_power(effect, target=0.80, alpha=ALPHA, paired=True):
    if not effect:
        return float("inf")
    zc, zb = _N.inv_cdf(1 - alpha / 2), _N.inv_cdf(target)
    k = 1 if paired else 2
    return int(np.ceil(k * ((zc + zb) / effect) ** 2))


def label(x, kind):
    a = abs(x)
    if kind == "rb":
        return "large" if a >= 0.5 else "medium" if a >= 0.3 else "small" if a >= 0.1 else "negligible"
    return "large" if a >= 0.8 else "medium" if a >= 0.5 else "small" if a >= 0.2 else "negligible"


def holm(pvals):
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj, running = [0.0] * m, 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj


# ---- collect paired metrics --------------------------------------------------
prop_metrics, cont_metrics = [], []

if "per_sample" in dir() and per_sample and per_sample[0]["sid"] in set(common):
    rows = {s["sid"]: s for s in per_sample}
    have_split = True
    print("Compiler metrics: reusing `per_sample` from the compiler cell above.")
else:
    rows, have_split = {}, False
    for sid in common:
        nm = json.loads((NAIVE_DIR / sid / "meta.json").read_text(encoding="utf-8")).get("validation", {})
        pm = json.loads((PIPELINE_DIR / sid / "meta.json").read_text(encoding="utf-8")).get("validation", {})
        rows[sid] = {"naive": nm, "pipeline": pm}
    print("Compiler metrics: loaded from meta.json (run the compiler cell for the "
          "syntax/semantic split).")

get = lambda sid, side, k, d=0: rows[sid][side].get(k, d)
prop_metrics.append(("Valid compile rate",
                     [bool(get(s, "naive", "is_valid", False)) for s in common],
                     [bool(get(s, "pipeline", "is_valid", False)) for s in common]))
cont_metrics.append(("Compiler errors",
                     [get(s, "naive", "error_count") for s in common],
                     [get(s, "pipeline", "error_count") for s in common], True))
if have_split:
    for nm_, key in [("Syntax-clean rate", "syntax_error_count"),
                     ("Semantic-clean rate", "semantic_error_count")]:
        prop_metrics.append((nm_,
                             [get(s, "naive", key) == 0 for s in common],
                             [get(s, "pipeline", key) == 0 for s in common]))

# Security metrics: in-memory results if the security cell ran this session,
# otherwise its checkpoints on disk.
_ns, _ps = None, None
if "naive_sec_results" in dir() and "pipe_sec_results" in dir():
    _ns, _ps = naive_sec_results, pipe_sec_results
    _src = "in memory"
else:
    _ckn = ROOT / "nl2solidity" / "dataset" / "security_results_naive.json"
    _ckp = ROOT / "nl2solidity" / "dataset" / "security_results_pipeline.json"
    if _ckn.exists() and _ckp.exists():
        _ns = json.loads(_ckn.read_text(encoding="utf-8"))
        _ps = json.loads(_ckp.read_text(encoding="utf-8"))
        _src = "from checkpoints"
if _ns is not None:
    sc = [s for s in common if s in _ns and s in _ps]
    if sc:
        tag = "" if len(sc) == n else f" (n={len(sc)})"
        print(f"Security metrics: {_src}, {len(sc)}/{n} paired samples.")
        prop_metrics.append((f"Actionable-clean rate{tag}",
                             [_ns[s].get("n_actionable", 0) == 0 for s in sc],
                             [_ps[s].get("n_actionable", 0) == 0 for s in sc]))
        cont_metrics.append((f"Actionable findings{tag}",
                             [_ns[s].get("n_actionable", 0) for s in sc],
                             [_ps[s].get("n_actionable", 0) for s in sc], True))
        cont_metrics.append((f"All findings{tag}",
                             [_ns[s].get("n_findings", 0) for s in sc],
                             [_ps[s].get("n_findings", 0) for s in sc], True))
    else:
        print("Security metrics: SKIPPED (no sample has results on both sides yet).")
else:
    print("Security metrics: SKIPPED (run the security cell above first).")

# Execution metrics: in-memory results if the execution cell ran this session,
# otherwise its checkpoints on disk.
_ne, _pe = None, None
if "naive_exec_results" in dir() and "pipe_exec_results" in dir():
    _ne, _pe = naive_exec_results, pipe_exec_results
    _src = "in memory"
else:
    _ckn = ROOT / "nl2solidity" / "dataset" / "exec_results_naive.json"
    _ckp = ROOT / "nl2solidity" / "dataset" / "exec_results_pipeline.json"
    if _ckn.exists() and _ckp.exists():
        _ne = json.loads(_ckn.read_text(encoding="utf-8"))
        _pe = json.loads(_ckp.read_text(encoding="utf-8"))
        _src = "from checkpoints"
if _ne is not None:
    ec = [s for s in common if s in _ne and s in _pe]
    if ec:
        tag = "" if len(ec) == n else f" (n={len(ec)})"
        print(f"Execution metrics: {_src}, {len(ec)}/{n} paired samples.")
        prop_metrics.append((f"Foundry fuzz-pass rate{tag}",
                             [bool(_ne[s].get("success")) for s in ec],
                             [bool(_pe[s].get("success")) for s in ec]))
        cont_metrics.append((f"Execution defects{tag}",
                             [_ne[s].get("n_failures", 0) for s in ec],
                             [_pe[s].get("n_failures", 0) for s in ec], True))
    else:
        print("Execution metrics: SKIPPED (no sample has results on both sides yet).")
else:
    print("Execution metrics: SKIPPED (run the execution cell above first).")

# Spec-alignment similarity: paired continuous, from the checkpoint.
if CKPT.exists():
    _st = json.loads(CKPT.read_text(encoding="utf-8"))
    pair_ids = []
    for sid in common:
        a = _st.get(sid, {}).get("similarity")
        b = (json.loads((PIPELINE_DIR / sid / "meta.json").read_text(encoding="utf-8"))
             .get("spec_alignment", {}).get("similarity"))
        if a is not None and b is not None:
            pair_ids.append((sid, a, b))
    if pair_ids:
        cont_metrics.append((f"Spec-alignment similarity (n={len(pair_ids)})",
                             [a for _, a, _ in pair_ids], [b for _, _, b in pair_ids], False))
    else:
        print("Spec alignment: SKIPPED (checkpoint has no scored pairs yet).")
else:
    print("Spec alignment: SKIPPED (no checkpoint; run the scoring cell above).")

print(f"\nMetrics assembled: {len(prop_metrics)} proportion, {len(cont_metrics)} continuous")


# ---- run the tests -----------------------------------------------------------
results = []

for name, a, b in prop_metrics:
    p1, p2 = float(np.mean(b)), float(np.mean(a))
    h = cohens_h(p1, p2)
    mc = mcnemar(a, b)
    results.append({
        "metric": name, "kind": "proportion", "n": len(a),
        "naive": p2 * 100, "pipeline": p1 * 100, "delta": (p1 - p2) * 100,
        "effect_name": "Cohen's h", "effect": h, "effect_label": label(h, "d"),
        "test": "McNemar" + (" (exact)" if mc["exact"] else ""),
        "stat": f"b={mc['b']} c={mc['c']}", "p": mc["p"],
        "power": power_mcnemar(mc["b"], mc["c"]),
        "power_alt": power_two_proportions(h, len(a)),
        "n_needed": n_for_power(h, paired=False), "detail": mc,
    })

for name, a, b, lower_better in cont_metrics:
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    diffs = b - a
    signed = -diffs if lower_better else diffs
    wc = wilcoxon(signed)
    dz = cohens_dz(signed)
    results.append({
        "metric": name, "kind": "continuous", "n": len(a),
        "naive": float(np.mean(a)), "pipeline": float(np.mean(b)),
        "delta": float(np.mean(b) - np.mean(a)),
        "median_delta": float(np.median(b) - np.median(a)),
        "effect_name": "Cohen's d_z", "effect": dz, "effect_label": label(dz, "d"),
        "rb": wc["rb"], "rb_label": label(wc["rb"], "rb"),
        "test": "Wilcoxon signed-rank",
        "stat": f"W+={wc['w_plus']:.0f} n={wc['n_nonzero']}", "p": wc["p"],
        "power": power_paired(dz, len(a)),
        "n_needed": n_for_power(dz, paired=True), "detail": wc,
    })

for r, adj in zip(results, holm([r["p"] for r in results])):
    r["p_holm"] = adj

print(f"\n{'='*118}")
print(f"POWER ANALYSIS & PAIRED SIGNIFICANCE TESTS   (n={n} paired samples, alpha={ALPHA})")
print("=" * 118)
print(f"{'Metric':<34}{'Naive':>9}{'Pipeline':>10}{'Delta':>9}   "
      f"{'Effect size':<32}{'p (Holm)':>12}{'Power':>10}")
print("-" * 118)
for r in results:
    unit = "%" if r["kind"] == "proportion" else ""
    eff = f"{r['effect_name']} = {r['effect']:+.3f} ({r['effect_label']})"
    pw = "> 0.9999" if r["power"] > 0.9999 else f"{r['power']:.4f}"
    print(f"{r['metric']:<34}{r['naive']:>8.2f}{unit:<1}{r['pipeline']:>9.2f}{unit:<1}"
          f"{r['delta']:>+8.2f}{unit:<1}  {eff:<32}{fmt_p(r['p_holm']):>12}{pw:>10}")
print("-" * 118)

underpowered = [r for r in results if r["power"] < 0.80]
print(f"\n{'='*118}")
if underpowered:
    print("Underpowered at 80% (effect too small for this n \u2014 report as inconclusive, "
          "not as 'no difference'):")
    for r in underpowered:
        print(f"  - {r['metric']}: power {r['power']:.3f}, needs n\u2248{r['n_needed']}")
else:
    print(f"Every metric above reaches >= 80% power at n={n}. "
          "Non-significant results here would be genuine nulls, not missing sensitivity.")


In [ ]:
# --- Effect size and study sensitivity ---------------------------------------
names = [r["metric"] for r in results]
signed = [r["effect"] for r in results]
needed = [r["n_needed"] for r in results]
have = [r["n"] for r in results]

def _vals(r):
    if r["kind"] == "proportion":
        return f"{r['naive']:.1f}% \u2192 {r['pipeline']:.1f}%  ({r['delta']:+.1f} pp)"
    return f"{r['naive']:.2f} \u2192 {r['pipeline']:.2f}  ({r['delta']:+.2f} mean)"

ylabels = [f"{r['metric']}\n{_vals(r)}" for r in results]
palette = plt.get_cmap("tab10")
cols = [palette(i % 10) for i in range(len(results))]
y = np.arange(len(results))

fig, axes = plt.subplots(1, 2, figsize=(15, max(4.5, 0.95 * len(results) + 1.5)))

ax = axes[0]
lim = max(1.0, max(abs(e) for e in signed) * 1.5)
for lo, hi, shade in ((0.2, 0.5, 0.06), (0.5, 0.8, 0.10), (0.8, lim, 0.15)):
    for sgn in (1, -1):
        ax.axvspan(sgn * lo, sgn * hi, color="grey", alpha=shade, lw=0)
for xv, lab in ((0.2, "small"), (0.5, "med"), (0.8, "large")):
    ax.text(xv, len(results) - 0.42, lab, fontsize=7, color="dimgrey",
            ha="center", va="bottom")
ax.barh(y, signed, color=cols, edgecolor="black", height=0.5)
for i, r in enumerate(results):
    off = 0.04 * lim if r["effect"] >= 0 else -0.04 * lim
    ax.text(r["effect"] + off, i, f"{r['effect']:+.2f}\n{r['effect_label']}",
            va="center", ha="left" if r["effect"] >= 0 else "right", fontsize=8)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(y, ylabels, fontsize=8)
ax.set_xlim(-lim, lim)
ax.set_xlabel("effect size   (Cohen's $h$ for rates, Cohen's $d_z$ for paired values)")
ax.set_title("How big is the difference?\n(right of 0 = pipeline better)", fontsize=10)
ax.invert_yaxis()

ax = axes[1]
plot_needed = [min(nd, 10 ** 6) if np.isfinite(nd) else 10 ** 6 for nd in needed]
ax.barh(y, plot_needed, color=cols, edgecolor="black", height=0.5, log=True)
for i, (nd, hv) in enumerate(zip(needed, have)):
    if not np.isfinite(nd):
        ax.text(2, i, "  effect \u2248 0: no sample size suffices", va="center", fontsize=8)
        continue
    ratio = hv / nd
    note = (f"  {nd} needed \u2014 you have {hv} ({ratio:.0f}\u00d7 margin)" if ratio >= 1
            else f"  {nd} needed \u2014 you have {hv} (UNDERPOWERED)")
    ax.text(nd * 1.15, i, note, va="center", fontsize=8)
ax.axvline(n, ls=":", color="black", lw=1.4)
ax.text(n * 1.15, -0.42, f"n = {n} collected", fontsize=8, va="top")
ax.set_yticks(y, ["" for _ in y])
ax.set_xscale("log")
ax.set_xlim(1, 10 ** 6)
ax.set_xlabel("paired samples needed for 80% power (log scale)")
ax.set_title(f"Was the study big enough?\n(bar \u226a dotted line = heavily powered)", fontsize=10)
ax.invert_yaxis()

plt.tight_layout()
plt.show()
